# 3.2 · Optimización de hiperparámetros (Optuna), de forma iterativa

Metas del notebook: **mejorar el AUC y el log loss de validación cruzada** con un XGBoost afinado, y terminar con un **modelo final** entrenado con las variables seleccionadas (notebook 3.1) y los mejores hiperparámetros.

Cómo se hace, paso a paso:

1. **Datos y variables** – variables seleccionadas en el notebook 3.1 (si existen).
2. **Punto de partida** – XGBoost con parámetros por defecto: AUC y log loss de validación cruzada.
3. **Rondas de afinación** – cada ronda busca con Optuna sobre un grupo de parámetros, **partiendo del mejor resultado anterior**:
   las rondas impares afinan la estructura de los árboles y las pares el *boosting* y la regularización (espacio en `schema/param_grid.json`).
   Una ronda solo se acepta si mejora el AUC; el proceso se detiene cuando la mejora es menor que `MIN_GAIN` o se llega a `MAX_ROUNDS`.
4. **Modelo final** – se entrena con todas las filas de `train`, las variables seleccionadas y los mejores parámetros, y se evalúa en `test`.

> El `test` de `model_nb` es una muestra distinta (sin ids compartidos) pero del mismo archivo que `train`, así que sirve para comparar configuraciones, no como validación independiente.
> Para producción, repite la búsqueda con todos los datos: `python train/3-train.py --param-search optuna`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", None)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Project root: notebooks live in <ROOT>/notebooks. Safe to re-run.
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.model_paths import MODEL_DIR, ReadPathsPreprocess, WritePathsFeatureSelection
from src.preprocessor_utils import get_column_names
from src.utils import _XGB_STAGE_PARAMS, optuna_optimizer_xgb_classifier

## Configuración y rutas

In [ ]:
MODEL_NAME = "model_nb"           # light model on a small sample
TARGET = "TARGET"
USE_SELECTED_FEATURES = True      # features selected in notebook 3.1

N_SPLITS = 5                      # cross-validation folds (same folds in every comparison)
SEED = 42
MAX_ROUNDS = 6                    # tuning rounds: 1 trees, 2 boosting, 3 trees, ...
MIN_GAIN = 0.001                  # stop when a round improves the CV AUC by less than this
N_TRIALS = 20                     # Optuna trials per round
TIMEOUT = 240                     # seconds per round

# Where this notebook leaves its results (everything under models/ is git-ignored)
METRICS_PATH = MODEL_DIR / MODEL_NAME / "metrics"
IMG_PATH = MODEL_DIR / MODEL_NAME / "img"
ARTIFACTS_PATH = MODEL_DIR / MODEL_NAME / "artifacts"
for d in (METRICS_PATH, IMG_PATH, ARTIFACTS_PATH):
    d.mkdir(parents=True, exist_ok=True)

BEST_PARAMS_PATH = METRICS_PATH / f"best_params_{MODEL_NAME}.json"
HISTORY_PATH = METRICS_PATH / f"tuning_history_{MODEL_NAME}.csv"
FINAL_METRICS_PATH = METRICS_PATH / f"final_metrics_{MODEL_NAME}.json"
FINAL_MODEL_PATH = ARTIFACTS_PATH / f"model_final_{MODEL_NAME}.joblib.dat"
HISTORY_IMG_PATH = IMG_PATH / f"tuning_rounds_{MODEL_NAME}.png"

## 1 · Datos y variables

Se usan las variables que dejó el selector (notebook 3.1) en `metrics/`. Si no existe ese archivo, se usan todas.

In [ ]:
rp = ReadPathsPreprocess(MODEL_NAME)
train = pd.read_csv(rp.train_dataset_path)
test = pd.read_csv(rp.test_dataset_path)

try:
    preprocessor = joblib.load(rp.preprocessor_path)
except AttributeError as exc:      # e.g. "module '__main__' has no attribute 'ManualSelector'"
    raise RuntimeError(
        f"{rp.preprocessor_path.name} was pickled with classes defined inside an old notebook. "
        "Restart the kernel and re-run notebook 2 to regenerate it."
    ) from exc
ALL_FEATURES = [c for c in get_column_names(preprocessor) if c in train.columns]

best_features_path = WritePathsFeatureSelection(MODEL_NAME).best_schema_path
if USE_SELECTED_FEATURES and best_features_path.exists():
    with open(best_features_path, encoding="utf-8") as f:
        FEATURES = [c for c in json.load(f) if c in train.columns]
    print(f"{len(FEATURES)} selected features (of {len(ALL_FEATURES)})")
else:
    FEATURES = ALL_FEATURES
    print(f"all {len(FEATURES)} features (no selector output found)")

X, y = train[FEATURES], train[TARGET]
print(f"train {X.shape}, test {test.shape}, default rate {y.mean():.2%}")

## 2 · Punto de partida

XGBoost con parámetros por defecto y **la misma validación cruzada** que se usará en todas las rondas. Las métricas son solo dos: AUC (más alto es mejor) y log loss (más bajo es mejor).

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def cv_scores(params, n_estimators=None):
    """Mean validation AUC and log loss over the CV folds for XGBoost with `params`."""
    extra = {"n_estimators": n_estimators} if n_estimators else {}
    model = xgb.XGBClassifier(**params, **extra, random_state=SEED, n_jobs=-1)
    res = cross_validate(model, X, y, cv=cv, scoring={"auc": "roc_auc", "logloss": "neg_log_loss"})
    return res["test_auc"].mean(), -res["test_logloss"].mean()


base_auc, base_logloss = cv_scores({})
print(f"default XGBoost   AUC {base_auc:.4f}   log loss {base_logloss:.4f}")
history = [{"round": 0, "tuned": "default", "auc": base_auc, "logloss": base_logloss,
            "gain_auc": np.nan, "accepted": True, "n_estimators": 100}]

## 3 · Rondas de afinación

En cada ronda:

1. Optuna (TPE + poda *Hyperband*) busca sobre el grupo de parámetros de la ronda, con el resto fijo en el **mejor resultado acumulado**.
2. El mejor resultado anterior entra como primer ensayo, así que una ronda nunca empieza peor de lo que ya se tiene.
3. El candidato se vuelve a puntuar con la validación cruzada de arriba (número de árboles fijo), y **solo se acepta si mejora el AUC**.

El valor interno de Optuna se calcula con parada temprana sobre el propio pliegue de validación, por eso es algo optimista; la tabla de abajo usa siempre la medición honesta.

In [ ]:
incumbent = {}                    # best parameters so far (merged across rounds)
best_n_estimators = None          # number of trees from early stopping
best_auc = base_auc

for round_ in range(1, MAX_ROUNDS + 1):
    stage = 1 if round_ % 2 == 1 else 2            # 1 = tree structure, 2 = boosting / regularization
    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED + round_),
        pruner=optuna.pruners.HyperbandPruner())
    warm = {k: v for k, v in incumbent.items() if k in _XGB_STAGE_PARAMS[stage]}
    if warm:
        study.enqueue_trial(warm)                  # start from the best we already have
    study.optimize(
        lambda trial: optuna_optimizer_xgb_classifier(
            trial, X=X, y=y, n_splits=N_SPLITS, random_state=SEED, base_params=incumbent, stage=stage),
        n_trials=N_TRIALS, timeout=TIMEOUT)

    candidate = {**incumbent, **study.best_params}
    candidate_n = study.best_trial.user_attrs["n_estimators"]
    auc, logloss = cv_scores(candidate, candidate_n)
    gain = auc - best_auc
    accepted = gain > 0
    if accepted:
        incumbent, best_n_estimators, best_auc = candidate, candidate_n, auc

    history.append({"round": round_, "tuned": "trees" if stage == 1 else "boosting",
                    "auc": auc, "logloss": logloss, "gain_auc": gain,
                    "accepted": accepted, "n_estimators": candidate_n})
    print(f"round {round_} ({'trees' if stage == 1 else 'boosting':8s}) AUC {auc:.4f}  log loss {logloss:.4f}  "
          f"gain {gain:+.4f}  {'accepted' if accepted else 'rejected'}")
    if gain < MIN_GAIN:
        print(f"Gain below {MIN_GAIN}: stopping.")
        break

history = pd.DataFrame(history)
history.to_csv(HISTORY_PATH, index=False)
history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, col, better in ((axes[0], "auc", "higher is better"), (axes[1], "logloss", "lower is better")):
    ax.plot(history["round"], history[col], "o-")
    ax.set_xlabel("round (0 = default)"); ax.set_title(f"CV {col} ({better})")
    ax.set_xticks(history["round"])
fig.tight_layout()
fig.savefig(HISTORY_IMG_PATH, dpi=200, bbox_inches="tight")
plt.show()

print("Best parameters so far:")
pd.Series({**incumbent, "n_estimators": best_n_estimators}, dtype=object)

## 4 · Modelo final

Con las variables seleccionadas y los mejores hiperparámetros, entrenado con **todas** las filas de `train` y evaluado en `test`.
La tabla separa el efecto de cada decisión: variables seleccionadas con parámetros por defecto, y luego los parámetros afinados.

In [ ]:
final_params = {**incumbent, "random_state": SEED, "n_jobs": -1}
if best_n_estimators:
    final_params["n_estimators"] = best_n_estimators

configs = {
    "all features, default params": (ALL_FEATURES, {"random_state": SEED, "n_jobs": -1}),
    "selected features, default params": (FEATURES, {"random_state": SEED, "n_jobs": -1}),
    "selected features, tuned params (final)": (FEATURES, final_params),
}
rows, models = [], {}
for label, (cols, params) in configs.items():
    model = xgb.XGBClassifier(**params).fit(train[cols], y)
    models[label] = model
    p_train, p_test = model.predict_proba(train[cols])[:, 1], model.predict_proba(test[cols])[:, 1]
    rows.append({"model": label, "n_features": len(cols),
                 "train_auc": roc_auc_score(y, p_train), "test_auc": roc_auc_score(test[TARGET], p_test),
                 "train_logloss": log_loss(y, p_train), "test_logloss": log_loss(test[TARGET], p_test)})
final_table = pd.DataFrame(rows).set_index("model")
final_table

In [ ]:
final_model = models["selected features, tuned params (final)"]
saved_params = {k: v for k, v in final_params.items() if k != "n_jobs"}

with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(saved_params, f, indent=2)
joblib.dump(final_model, FINAL_MODEL_PATH)
with open(FINAL_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump({"features": FEATURES, "cv_auc": best_auc,
               "metrics": final_table.loc["selected features, tuned params (final)"].to_dict()}, f, indent=2)

print(f"Best params   : {BEST_PARAMS_PATH}")
print(f"Final model   : {FINAL_MODEL_PATH}")
print(f"Final metrics : {FINAL_METRICS_PATH}")
print(f"\nReuse the parameters in the script pipeline (trains on all features of model_00):\n"
      f"  python train/3-train.py --params-file {BEST_PARAMS_PATH.relative_to(ROOT_DIR)}")
print("Next: notebook 4 explains this model with SHAP values.")